In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# MY CODE


In [2]:
!pip install transformers peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
os.environ["WANDB_PROJECT"] = "23f3000843-t22026"
os.environ["WANDB_ENTITY"] = "varnitchourasiya27-indian-institute-of-technology-madras"

In [4]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset
import wandb
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

options = ['A', 'B', 'C', 'D', 'E']

secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: varnitchourasiya27 (varnitchourasiya27-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (2000, 8)
Test shape : (500, 7)


In [6]:
def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row['answer']
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)

def evaluate_and_log(model_name, predict_fn, sample_size=200):
    run = wandb.init(
        entity="varnitchourasiya27-indian-institute-of-technology-madras",
        project="23f3000843-t22026",
        name=model_name,
        config={"model": model_name, "sample_size": sample_size}
    )
    sample = train.sample(sample_size, random_state=42)
    score = map_at_3(sample, predict_fn)
    wandb.log({"MAP@3": score})
    print(f"{model_name} → Local MAP@3: {score:.4f}")
    wandb.finish()
    return score

In [7]:
def format_input(row):
    return f"""Question: {row['prompt']}
A: {row['A']}
B: {row['B']}
C: {row['C']}
D: {row['D']}
E: {row['E']}
The best answer is:"""

def format_output(row):
    return row['answer']

# Test it
print("Input:")
print(format_input(train.iloc[0]))
print("\nOutput:")
print(format_output(train.iloc[0]))

Input:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.
D: Martin Heidegger believes that the rel

In [8]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_input_length=512, max_output_length=10):
        self.df = df
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_text = format_input(row)
        output_text = format_output(row)

        input_enc = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        output_enc = self.tokenizer(
            output_text,
            max_length=self.max_output_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        labels = output_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels': labels
        }

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load base model
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                    # rank of LoRA matrices
    lora_alpha=32,           # scaling factor
    lora_dropout=0.1,        # dropout for regularization
    target_modules=['q', 'v'] # which layers to apply LoRA to
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [10]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

train_dataset = MCQDataset(train_df, tokenizer)
val_dataset = MCQDataset(val_df, tokenizer)

Train size: 1800
Val size: 200


In [11]:
training_args = TrainingArguments(
    output_dir='./flan-t5-lora-mcq',
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='wandb',
    run_name='flan-t5-base-lora',
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,18.670331,17.610781
2,16.955837,15.342036
3,15.760631,13.654432
4,14.603690,12.761167
5,14.052169,12.254385
6,13.719850,12.081582
7,13.645577,12.022656


TrainOutput(global_step=791, training_loss=15.406497149943702, metrics={'train_runtime': 915.5495, 'train_samples_per_second': 13.762, 'train_steps_per_second': 0.864, 'total_flos': 8696433947443200.0, 'train_loss': 15.406497149943702, 'epoch': 7.0})

In [13]:
def predict_top3_finetuned(row):
    input_text = format_input(row)
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    model.to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        num_beams=5,
        num_return_sequences=3,
        early_stopping=True
    )
    
    predictions = []
    for output in outputs:
        answer = tokenizer.decode(output, skip_special_tokens=True).strip().upper()
        if answer in options and answer not in predictions:
            predictions.append(answer)
    
    for opt in options:
        if opt not in predictions:
            predictions.append(opt)
    
    return ' '.join(predictions[:3])

In [14]:
sample = train.sample(500, random_state=42)
score = map_at_3(sample, predict_top3_finetuned)
print(f"Fine-tuned Flan-T5-base LoRA Local MAP@3: {score:.4f}")

Fine-tuned Flan-T5-base LoRA Local MAP@3: 0.3783
